# Crypto 50x Leverage Backtest - FinRL Tutorial

50배 레버리지 암호화폐 트레이딩 백테스트

**Features:**
- 50x Leverage Long/Short Trading
- Liquidation Logic (청산)
- Trailing Stop Loss
- Position Visualization

**Prerequisites**: Run Part 1 (Data) notebook first to generate `crypto_5m_data.npz`

# Part 1. Import Packages

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from stable_baselines3 import A2C, PPO, DDPG, TD3, SAC

from finrl.config import TRAINED_MODEL_DIR

import gymnasium as gym
from gymnasium import spaces

print("Packages loaded!")

Packages loaded!


# Part 2. Load Test Data

In [2]:
# Load data from Part 1
data = np.load('crypto_5m_data.npz', allow_pickle=True)

test_price = data['test_price']
test_tech = data['test_tech']
crypto_pairs = data['crypto_pairs']

print(f"Test data: {len(test_price)} samples")
print(f"Number of coins: {test_price.shape[1]}")
print(f"Coin list: {crypto_pairs}")

Test data: 1706 samples
Number of coins: 5
Coin list: ['BTC/USDT' 'ETH/USDT' 'BNB/USDT' 'XRP/USDT' 'SOL/USDT']


# Part 3. 50x Leverage Environment

In [10]:
class CryptoLeverageEnv:
    """
    50x Leverage Cryptocurrency Trading Environment
    
    Features:
    - Long/Short positions with leverage
    - Liquidation when margin ratio falls below threshold
    - Trailing stop loss
    - Transaction costs (taker fee)
    
    Action Space: [-1, 1] per coin
    - Positive: Long position (action * leverage * capital allocation)
    - Negative: Short position
    - Zero: No position / Close position
    """
    
    def __init__(
        self,
        config,
        lookback=1,
        initial_capital=1e6,
        leverage=50,
        taker_fee=0.0004,  # 0.04% taker fee (Binance futures)
        maintenance_margin=0.005,  # 0.5% maintenance margin for 50x
        trailing_stop_pct=0.02,  # 2% trailing stop
        use_trailing_stop=True,
        gamma=0.99,
    ):
        self.lookback = lookback
        self.initial_capital = initial_capital
        self.leverage = leverage
        self.taker_fee = taker_fee
        self.maintenance_margin = maintenance_margin
        self.trailing_stop_pct = trailing_stop_pct
        self.use_trailing_stop = use_trailing_stop
        self.gamma = gamma
        
        self.price_array = config["price_array"]
        self.tech_array = config["tech_array"]
        
        self.crypto_num = self.price_array.shape[1]
        self.max_step = self.price_array.shape[0] - lookback - 1
        
        # Environment info
        self.env_name = "CryptoLeverageEnv"
        self.state_dim = (
            1 +  # cash ratio
            self.crypto_num +  # position sizes
            self.crypto_num +  # unrealized PnL
            (self.price_array.shape[1] + self.tech_array.shape[1]) * lookback
        )
        self.action_dim = self.crypto_num
        self.if_discrete = False
        
        # Tracking lists for visualization
        self.history = []
        
        self.reset()
        self.state_dim = self.get_state().shape[0] # 실제 생성된 state의 길이를 측정
    
    def reset(self, *, seed=None, options=None):
        self.time = self.lookback - 1
        self.cash = self.initial_capital
        self.current_price = self.price_array[self.time]
        self.current_tech = self.tech_array[self.time]
        
        # Position tracking
        # positions: positive = long, negative = short (in notional value)
        self.positions = np.zeros(self.crypto_num, dtype=np.float32)
        self.entry_prices = np.zeros(self.crypto_num, dtype=np.float32)
        self.highest_prices = np.zeros(self.crypto_num, dtype=np.float32)  # for trailing stop (long)
        self.lowest_prices = np.full(self.crypto_num, np.inf, dtype=np.float32)  # for trailing stop (short)
        
        # PnL tracking
        self.unrealized_pnl = np.zeros(self.crypto_num, dtype=np.float32)
        self.realized_pnl = 0.0
        
        self.total_asset = self.cash
        self.episode_return = 0.0
        self.gamma_return = 0.0
        
        # History for visualization
        self.history = [{
            'time': self.time,
            'cash': self.cash,
            'total_asset': self.total_asset,
            'positions': self.positions.copy(),
            'prices': self.current_price.copy(),
            'action': None,
            'liquidated': [],
            'trailing_stopped': [],
        }]
        
        return self.get_state()
    
    def _calculate_unrealized_pnl(self):
        """Calculate unrealized PnL for all positions"""
        pnl = np.zeros(self.crypto_num, dtype=np.float32)
        for i in range(self.crypto_num):
            if self.positions[i] != 0 and self.entry_prices[i] > 0:
                price_change = (self.current_price[i] - self.entry_prices[i]) / self.entry_prices[i]
                # Long: profit when price goes up
                # Short: profit when price goes down
                if self.positions[i] > 0:  # Long
                    pnl[i] = abs(self.positions[i]) * price_change
                else:  # Short
                    pnl[i] = abs(self.positions[i]) * (-price_change)
        return pnl
    
    def _check_liquidation(self):
        """Check and execute liquidation for positions below maintenance margin"""
        liquidated = []
        for i in range(self.crypto_num):
            if self.positions[i] != 0:
                position_value = abs(self.positions[i])
                margin_used = position_value / self.leverage
                
                # Calculate loss
                if self.positions[i] > 0:  # Long
                    loss_pct = (self.entry_prices[i] - self.current_price[i]) / self.entry_prices[i]
                else:  # Short
                    loss_pct = (self.current_price[i] - self.entry_prices[i]) / self.entry_prices[i]
                
                # Check if loss exceeds margin - maintenance margin
                max_loss_pct = (1 / self.leverage) - self.maintenance_margin
                
                if loss_pct >= max_loss_pct:
                    # Liquidation! Lose the margin
                    liquidation_loss = margin_used
                    self.cash -= liquidation_loss
                    self.realized_pnl -= liquidation_loss
                    
                    liquidated.append({
                        'coin': i,
                        'position': self.positions[i],
                        'entry_price': self.entry_prices[i],
                        'liquidation_price': self.current_price[i],
                        'loss': liquidation_loss
                    })
                    
                    # Reset position
                    self.positions[i] = 0
                    self.entry_prices[i] = 0
                    self.highest_prices[i] = 0
                    self.lowest_prices[i] = np.inf
        
        return liquidated
    
    def _check_trailing_stop(self):
        """Check and execute trailing stop loss"""
        if not self.use_trailing_stop:
            return []
        
        stopped = []
        for i in range(self.crypto_num):
            if self.positions[i] != 0:
                triggered = False
                
                if self.positions[i] > 0:  # Long position
                    # Update highest price
                    self.highest_prices[i] = max(self.highest_prices[i], self.current_price[i])
                    # Check trailing stop
                    drop_pct = (self.highest_prices[i] - self.current_price[i]) / self.highest_prices[i]
                    if drop_pct >= self.trailing_stop_pct:
                        triggered = True
                
                else:  # Short position
                    # Update lowest price
                    self.lowest_prices[i] = min(self.lowest_prices[i], self.current_price[i])
                    # Check trailing stop
                    rise_pct = (self.current_price[i] - self.lowest_prices[i]) / self.lowest_prices[i]
                    if rise_pct >= self.trailing_stop_pct:
                        triggered = True
                
                if triggered:
                    # Close position at current price
                    pnl = self._close_position(i)
                    stopped.append({
                        'coin': i,
                        'position': self.positions[i],
                        'pnl': pnl
                    })
        
        return stopped
    
    def _close_position(self, coin_idx):
        """Close a position and return realized PnL"""
        if self.positions[coin_idx] == 0:
            return 0.0
        
        position_value = abs(self.positions[coin_idx])
        margin_used = position_value / self.leverage
        
        # Calculate PnL
        if self.positions[coin_idx] > 0:  # Long
            price_change = (self.current_price[coin_idx] - self.entry_prices[coin_idx]) / self.entry_prices[coin_idx]
        else:  # Short
            price_change = (self.entry_prices[coin_idx] - self.current_price[coin_idx]) / self.entry_prices[coin_idx]
        
        pnl = position_value * price_change
        
        # Apply closing fee
        fee = position_value * self.taker_fee
        pnl -= fee
        
        # Update cash and realized PnL
        self.cash += margin_used + pnl
        self.realized_pnl += pnl
        
        # Reset position
        self.positions[coin_idx] = 0
        self.entry_prices[coin_idx] = 0
        self.highest_prices[coin_idx] = 0
        self.lowest_prices[coin_idx] = np.inf
        
        return pnl
    
    def _open_position(self, coin_idx, action):
        """Open a new position"""
        if action == 0:
            return
        
        # Calculate position size
        # action is in [-1, 1], represents fraction of available capital to use
        available_capital = self.cash * 0.95  # Keep 5% as buffer
        margin_to_use = available_capital * abs(action) / self.crypto_num
        
        if margin_to_use < 100:  # Minimum position size
            return
        
        position_value = margin_to_use * self.leverage
        
        # Apply opening fee
        fee = position_value * self.taker_fee
        
        # Deduct margin and fee from cash
        self.cash -= (margin_to_use + fee)
        
        # Set position (positive for long, negative for short)
        if action > 0:
            self.positions[coin_idx] = position_value
        else:
            self.positions[coin_idx] = -position_value
        
        self.entry_prices[coin_idx] = self.current_price[coin_idx]
        self.highest_prices[coin_idx] = self.current_price[coin_idx]
        self.lowest_prices[coin_idx] = self.current_price[coin_idx]
    
    def step(self, actions):
        self.time += 1
        self.current_price = self.price_array[self.time]
        self.current_tech = self.tech_array[self.time]
        
        # Check liquidations first
        liquidated = self._check_liquidation()
        
        # Check trailing stops
        trailing_stopped = self._check_trailing_stop()
        
        # Process actions for each coin
        for i in range(self.crypto_num):
            action = actions[i]
            
            # Skip if position was liquidated or stopped
            if any(l['coin'] == i for l in liquidated):
                continue
            if any(s['coin'] == i for s in trailing_stopped):
                continue
            
            current_position = self.positions[i]
            
            # Determine action type
            if current_position == 0:
                # No position - open new if action is significant
                if abs(action) > 0.1:
                    self._open_position(i, action)
            else:
                # Has position
                if current_position > 0:  # Long
                    if action < -0.1:  # Want to go short - close long first
                        self._close_position(i)
                        if abs(action) > 0.3:  # Strong signal - open short
                            self._open_position(i, action)
                    elif action < 0.1:  # Weak signal - close position
                        self._close_position(i)
                else:  # Short
                    if action > 0.1:  # Want to go long - close short first
                        self._close_position(i)
                        if abs(action) > 0.3:  # Strong signal - open long
                            self._open_position(i, action)
                    elif action > -0.1:  # Weak signal - close position
                        self._close_position(i)
        
        # Calculate unrealized PnL
        self.unrealized_pnl = self._calculate_unrealized_pnl()
        
        # Calculate total asset
        margin_in_positions = sum(
            abs(pos) / self.leverage for pos in self.positions if pos != 0
        )
        prev_total_asset = self.total_asset
        self.total_asset = self.cash + margin_in_positions + self.unrealized_pnl.sum()
        
        # Calculate reward
        reward = (self.total_asset - prev_total_asset) * 2**-16
        
        # Check if done
        done = self.time == self.max_step or self.total_asset < self.initial_capital * 0.1
        
        self.gamma_return = self.gamma_return * self.gamma + reward
        
        if done:
            reward = self.gamma_return
            self.episode_return = self.total_asset / self.initial_capital
        
        # Record history
        self.history.append({
            'time': self.time,
            'cash': self.cash,
            'total_asset': self.total_asset,
            'positions': self.positions.copy(),
            'prices': self.current_price.copy(),
            'unrealized_pnl': self.unrealized_pnl.copy(),
            'action': actions.copy(),
            'liquidated': liquidated,
            'trailing_stopped': trailing_stopped,
        })
        
        state = self.get_state()
        return state, reward, done, None
    
    def get_state(self):
        """Get current state"""
        # Normalize values
        cash_ratio = self.cash / self.initial_capital
        position_ratios = self.positions / (self.initial_capital * self.leverage) * 100
        pnl_ratios = self.unrealized_pnl / self.initial_capital * 100
        
        state = np.hstack([
            [cash_ratio],
            position_ratios,
            pnl_ratios,
        ])
        
        # Add technical indicators
        for i in range(self.lookback):
            tech_i = self.tech_array[self.time - i]
            normalized_tech_i = tech_i * 2**-15
            state = np.hstack((state, normalized_tech_i))
        
        return state.astype(np.float32)
    
    def close(self):
        pass

print("CryptoLeverageEnv class defined!")

CryptoLeverageEnv class defined!


# Part 4. Gym Wrapper

In [9]:
class CryptoLeverageGym(gym.Env):
    """Gymnasium wrapper for CryptoLeverageEnv"""
    
    def __init__(self, config, **kwargs):
        super().__init__()
        self.env = CryptoLeverageEnv(config, **kwargs)
        
        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf,
            shape=(self.env.state_dim,), dtype=np.float32
        )
        self.action_space = spaces.Box(
            low=-1, high=1,
            shape=(self.env.action_dim,), dtype=np.float32
        )
    
    def reset(self, seed=None, options=None):
        state = self.env.reset(seed=seed, options=options)
        return state, {}
    
    def step(self, action):
        state, reward, done, info = self.env.step(action)
        return state, reward, done, False, info or {}

# Create test environment
test_config = {
    "price_array": test_price,
    "tech_array": test_tech,
}

env_test = CryptoLeverageGym(
    config=test_config,
    lookback=1,
    initial_capital=1_000_000,
    leverage=50,
    taker_fee=0.0004,
    trailing_stop_pct=0.02,
    use_trailing_stop=True,
)

print(f"Test environment created!")
print(f"Max Steps: {env_test.env.max_step}")
print(f"State Dim: {env_test.env.state_dim}")
print(f"Action Dim: {env_test.env.action_dim}")
print(f"Leverage: {env_test.env.leverage}x")

Test environment created!
Max Steps: 1704
State Dim: 51
Action Dim: 5
Leverage: 50x


# Part 5. Backtest Function

In [5]:
def run_leverage_backtest(model, env_config, model_name="Model", **env_kwargs):
    """Run backtest with leverage environment"""
    
    # Create new environment
    env = CryptoLeverageGym(config=env_config, **env_kwargs)
    
    # Initialize
    obs, _ = env.reset()
    done = False
    
    step = 0
    while not done:
        # Predict action
        action, _ = model.predict(obs, deterministic=True)
        
        # Step
        obs, reward, done, truncated, info = env.step(action)
        
        step += 1
        if step % 500 == 0:
            current_value = env.env.total_asset
            print(f"{model_name} - Step {step}: Asset = ${current_value:,.0f}")
    
    # Get history from environment
    history = env.env.history
    
    # Create result DataFrames
    df_account = pd.DataFrame({
        'step': [h['time'] for h in history],
        'account_value': [h['total_asset'] for h in history],
        'cash': [h['cash'] for h in history],
    })
    
    # Count liquidations and trailing stops
    total_liquidations = sum(len(h['liquidated']) for h in history)
    total_trailing_stops = sum(len(h['trailing_stopped']) for h in history)
    
    # Final stats
    final_return = (history[-1]['total_asset'] / history[0]['total_asset'] - 1) * 100
    print(f"\n{model_name} Results:")
    print(f"  Final Return: {final_return:.2f}%")
    print(f"  Final Asset: ${history[-1]['total_asset']:,.0f}")
    print(f"  Liquidations: {total_liquidations}")
    print(f"  Trailing Stops: {total_trailing_stops}")
    
    return df_account, history

print("Backtest function defined!")

Backtest function defined!


# Part 6. Load Models & Run Backtest

In [6]:
# Load models
models = {}

try:
    models['A2C'] = A2C.load(TRAINED_MODEL_DIR + "/crypto_a2c")
    print("A2C model loaded")
except:
    print("A2C model not found")

try:
    models['PPO'] = PPO.load(TRAINED_MODEL_DIR + "/crypto_ppo")
    print("PPO model loaded")
except:
    print("PPO model not found")

print(f"\nLoaded models: {list(models.keys())}")

A2C model loaded
PPO model loaded

Loaded models: ['A2C', 'PPO']


In [7]:
# Run backtest for each model
results = {}

env_kwargs = {
    'lookback': 1,
    'initial_capital': 1_000_000,
    'leverage': 50,
    'taker_fee': 0.0004,
    'trailing_stop_pct': 0.02,
    'use_trailing_stop': True,
}

for name, model in models.items():
    print(f"\n{'='*50}")
    print(f"{name} Backtest (50x Leverage)")
    print(f"{'='*50}")
    
    df_account, history = run_leverage_backtest(
        model, test_config, name, **env_kwargs
    )
    results[name] = {
        'account': df_account,
        'history': history
    }


A2C Backtest (50x Leverage)


ValueError: Error: Unexpected observation shape (46,) for Box environment, please use (41,) or (n_env, 41) for the observation shape.

# Part 7. Visualization

In [ ]:
def plot_backtest_results(results, crypto_pairs, figsize=(16, 12)):
    """Plot comprehensive backtest results with liquidation and trailing stop markers"""
    
    fig, axes = plt.subplots(3, 1, figsize=figsize)
    colors = {'A2C': 'blue', 'PPO': 'green', 'DDPG': 'orange', 'TD3': 'purple', 'SAC': 'brown'}
    
    # Plot 1: Portfolio Value
    ax1 = axes[0]
    for name, data in results.items():
        ax1.plot(data['account']['step'], data['account']['account_value'],
                 label=name, color=colors.get(name, 'gray'), linewidth=2)
        
        # Mark liquidations
        history = data['history']
        for h in history:
            for liq in h['liquidated']:
                ax1.scatter(h['time'], h['total_asset'], 
                           marker='x', color='red', s=100, zorder=5)
            for stop in h['trailing_stopped']:
                ax1.scatter(h['time'], h['total_asset'],
                           marker='o', color='yellow', s=50, zorder=5, edgecolors='black')
    
    ax1.axhline(y=1_000_000, color='gray', linestyle='--', alpha=0.5, label='Initial Capital')
    ax1.set_xlabel('Step (5min)')
    ax1.set_ylabel('Portfolio Value ($)')
    ax1.set_title('50x Leverage Crypto Trading - Portfolio Value\n(X: Liquidation, O: Trailing Stop)')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Position Heatmap for first model
    ax2 = axes[1]
    if results:
        first_model = list(results.keys())[0]
        history = results[first_model]['history']
        
        # Extract position data
        steps = [h['time'] for h in history]
        positions = np.array([h['positions'] for h in history])
        
        # Normalize for visualization
        max_pos = max(abs(positions.min()), abs(positions.max()))
        if max_pos > 0:
            positions_norm = positions / max_pos
        else:
            positions_norm = positions
        
        im = ax2.imshow(positions_norm.T, aspect='auto', cmap='RdYlGn',
                        vmin=-1, vmax=1, extent=[steps[0], steps[-1], -0.5, len(crypto_pairs)-0.5])
        ax2.set_yticks(range(len(crypto_pairs)))
        ax2.set_yticklabels([p.replace('/USDT', '') for p in crypto_pairs])
        ax2.set_xlabel('Step (5min)')
        ax2.set_ylabel('Coin')
        ax2.set_title(f'{first_model} Position Heatmap (Green: Long, Red: Short)')
        plt.colorbar(im, ax=ax2, label='Position Direction')
    
    # Plot 3: Individual Crypto Prices
    ax3 = axes[2]
    for i, pair in enumerate(crypto_pairs):
        prices = test_price[:, i]
        normalized = prices / prices[0] * 100
        ax3.plot(normalized, label=pair.replace('/USDT', ''), alpha=0.8)
    
    ax3.axhline(y=100, color='gray', linestyle='--', alpha=0.5)
    ax3.set_xlabel('Step (5min)')
    ax3.set_ylabel('Normalized Price (Start=100)')
    ax3.set_title('Individual Crypto Price Movement')
    ax3.legend(loc='upper right')
    ax3.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# Run visualization
if results:
    plot_backtest_results(results, crypto_pairs)

In [ ]:
def plot_detailed_positions(results, crypto_pairs, model_name=None):
    """Plot detailed position chart for a specific model"""
    
    if model_name is None:
        model_name = list(results.keys())[0]
    
    if model_name not in results:
        print(f"Model {model_name} not found")
        return
    
    history = results[model_name]['history']
    n_coins = len(crypto_pairs)
    
    fig, axes = plt.subplots(n_coins, 1, figsize=(16, 3*n_coins), sharex=True)
    
    for i, (ax, pair) in enumerate(zip(axes, crypto_pairs)):
        steps = [h['time'] for h in history]
        prices = [h['prices'][i] for h in history]
        positions = [h['positions'][i] for h in history]
        
        # Normalize price
        prices_norm = np.array(prices) / prices[0] * 100
        
        # Plot price
        ax.plot(steps, prices_norm, 'b-', label='Price', linewidth=1.5)
        
        # Color background based on position
        for j in range(len(steps)-1):
            if positions[j] > 0:  # Long
                ax.axvspan(steps[j], steps[j+1], alpha=0.3, color='green')
            elif positions[j] < 0:  # Short
                ax.axvspan(steps[j], steps[j+1], alpha=0.3, color='red')
        
        # Mark liquidations and trailing stops
        for h in history:
            for liq in h['liquidated']:
                if liq['coin'] == i:
                    price_idx = history.index(h)
                    ax.scatter(h['time'], prices_norm[price_idx],
                              marker='x', color='red', s=200, zorder=10, linewidths=3)
            for stop in h['trailing_stopped']:
                if stop['coin'] == i:
                    price_idx = history.index(h)
                    ax.scatter(h['time'], prices_norm[price_idx],
                              marker='o', color='yellow', s=100, zorder=10, edgecolors='black')
        
        ax.set_ylabel(f'{pair.replace("/USDT", "")}\nPrice (Norm)')
        ax.axhline(y=100, color='gray', linestyle='--', alpha=0.5)
        ax.grid(True, alpha=0.3)
        ax.legend(loc='upper right')
    
    axes[-1].set_xlabel('Step (5min)')
    plt.suptitle(f'{model_name} - Detailed Positions (Green: Long, Red: Short, X: Liquidation, O: Trailing Stop)',
                 fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

# Run detailed visualization
if results:
    for model_name in results.keys():
        plot_detailed_positions(results, crypto_pairs, model_name)

# Part 8. Summary Statistics

In [ ]:
def calculate_stats(results):
    """Calculate comprehensive statistics for each model"""
    
    summary = []
    
    for name, data in results.items():
        account = data['account']['account_value']
        history = data['history']
        
        # Basic stats
        initial_value = account.iloc[0]
        final_value = account.iloc[-1]
        total_return = (final_value / initial_value - 1) * 100
        
        # Max drawdown
        peak = account.expanding(min_periods=1).max()
        drawdown = (account - peak) / peak * 100
        max_drawdown = drawdown.min()
        
        # Sharpe ratio (simplified, using returns)
        returns = account.pct_change().dropna()
        sharpe = returns.mean() / returns.std() * np.sqrt(252 * 24 * 12) if returns.std() > 0 else 0
        
        # Liquidation & trailing stop counts
        total_liquidations = sum(len(h['liquidated']) for h in history)
        total_trailing_stops = sum(len(h['trailing_stopped']) for h in history)
        
        summary.append({
            'Model': name,
            'Final Value ($)': f"{final_value:,.0f}",
            'Total Return (%)': f"{total_return:.2f}",
            'Max Drawdown (%)': f"{max_drawdown:.2f}",
            'Sharpe Ratio': f"{sharpe:.2f}",
            'Liquidations': total_liquidations,
            'Trailing Stops': total_trailing_stops,
        })
    
    return pd.DataFrame(summary)

if results:
    df_summary = calculate_stats(results)
    print("\n" + "="*80)
    print("50x Leverage Backtest Results Summary")
    print("="*80)
    print(df_summary.to_string(index=False))

# Part 9. Compare with No Leverage

In [ ]:
# Run backtest without leverage (1x) for comparison
results_no_leverage = {}

env_kwargs_1x = {
    'lookback': 1,
    'initial_capital': 1_000_000,
    'leverage': 1,  # No leverage
    'taker_fee': 0.001,  # Spot fee
    'trailing_stop_pct': 0.05,  # Wider stop for spot
    'use_trailing_stop': False,
}

print("Running 1x (No Leverage) Backtest for Comparison...")

for name, model in models.items():
    print(f"\n{name} (1x):")
    df_account, history = run_leverage_backtest(
        model, test_config, f"{name} (1x)", **env_kwargs_1x
    )
    results_no_leverage[f"{name} (1x)"] = {
        'account': df_account,
        'history': history
    }

Running 1x (No Leverage) Backtest for Comparison...

A2C (1x):


ValueError: Error: Unexpected observation shape (46,) for Box environment, please use (41,) or (n_env, 41) for the observation shape.

In [ ]:
# Combined comparison plot
plt.figure(figsize=(14, 6))

# 50x Leverage results
for name, data in results.items():
    plt.plot(data['account']['step'], data['account']['account_value'],
             label=f"{name} (50x)", linewidth=2)

# 1x results
for name, data in results_no_leverage.items():
    plt.plot(data['account']['step'], data['account']['account_value'],
             label=name, linestyle='--', linewidth=2)

plt.axhline(y=1_000_000, color='gray', linestyle=':', alpha=0.5, label='Initial Capital')
plt.xlabel('Step (5min)')
plt.ylabel('Portfolio Value ($)')
plt.title('Leverage Comparison: 50x vs 1x (No Leverage)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

C:\Users\ANSL\AppData\Local\Temp\ipykernel_640692\21315299.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
# Combined summary
all_results = {**results, **results_no_leverage}
df_all_summary = calculate_stats(all_results)

print("\n" + "="*80)
print("Complete Comparison Summary")
print("="*80)
print(df_all_summary.to_string(index=False))


Complete Comparison Summary
Empty DataFrame
Columns: []
Index: []
